In [16]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [17]:
# 1. Load Data
# Dataset Anda menggunakan delimiter ';'
df = pd.read_csv('student_data.csv', sep=';')

In [18]:
# 2. Preprocessing
# Pisahkan Fitur (X) dan Target (y)
X = df.drop('Target', axis=1).values
y_raw = df['Target'].values

In [19]:
# Encode Target (Dropout, Enrolled, Graduate -> 0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(y_raw)
num_classes = len(np.unique(y))

# Scaling (Sangat PENTING untuk Neural Network)
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [20]:
# 3. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Hitung Class Weights untuk menangani ketidakseimbangan data
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# 5. Convert ke PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [21]:
# 6. Buat DataLoader
batch_size = 32

train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Data Loaded. Features: {X.shape[1]}, Classes: {num_classes}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Data Loaded. Features: 36, Classes: 3
Device: cpu


In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score

# --- FIX: Definisikan Variabel Penting Terlebih Dahulu ---
# Pastikan 'X_train' dan 'y' sudah ada dari tahap Persiapan Data
input_dim = X_train.shape[1] 
output_dim = len(np.unique(y))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Input Dim: {input_dim}, Output Dim: {output_dim}, Device: {device}")

# --- Definisi Model Neural ODE ---
# Kita gunakan implementasi Euler sederhana agar tidak perlu install 'torchdiffeq'
# Ini lebih stabil dan tidak memerlukan dependency tambahan yang ribet.

class ODEFunc(nn.Module):
    def __init__(self, dim):
        super(ODEFunc, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(), # Tanh sering lebih stabil untuk ODE daripada ReLU
            nn.Linear(dim, dim)
        )

    def forward(self, t, x):
        return self.net(x)

class TabularNeuralODE(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64):
        super(TabularNeuralODE, self).__init__()
        
        # Proyeksi fitur input ke hidden state
        self.feature_layer = nn.Linear(input_dim, hidden_dim)
        
        # Fungsi ODE (Turunan terhadap waktu)
        self.ode_func = ODEFunc(hidden_dim)
        
        # Classifier akhir
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # 1. Proyeksi awal
        h = self.feature_layer(x)
        
        # 2. Solve ODE (Integrasi Euler Manual)
        # Kita simulasikan perubahan fitur dari t=0 ke t=1
        steps = 6 # Jumlah langkah integrasi (semakin banyak semakin akurat tapi lambat)
        dt = 1.0 / steps
        
        for _ in range(steps):
            # h(t+dt) = h(t) + dt * f(h(t), t)
            h = h + dt * self.ode_func(0, h)
        
        # 3. Klasifikasi
        out = self.fc(h)
        return out

# --- Setup Training Neural ODE ---
model_ode = TabularNeuralODE(input_dim, output_dim, hidden_dim=64).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model_ode.parameters(), lr=0.005, weight_decay=1e-3)

best_f1 = 0
patience = 10
trigger_times = 0
epochs = 50 

print("\nMulai Training Neural ODE (Fixed)...")

for epoch in range(epochs):
    model_ode.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_ode(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    # Evaluasi
    model_ode.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_ode(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        trigger_times = 0
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early stopping di epoch {epoch+1}")
            break

print(f"Training Selesai. Best F1 Score Neural ODE: {best_f1:.4f}")

Input Dim: 36, Output Dim: 3, Device: cpu

Mulai Training Neural ODE (Fixed)...
Epoch 1/50 | Loss: 0.7651 | Val Acc: 0.7401 | Val F1: 0.7411
Epoch 2/50 | Loss: 0.7092 | Val Acc: 0.6791 | Val F1: 0.7035
Epoch 3/50 | Loss: 0.6937 | Val Acc: 0.7537 | Val F1: 0.7564
Epoch 4/50 | Loss: 0.6775 | Val Acc: 0.7367 | Val F1: 0.7460
Epoch 5/50 | Loss: 0.6773 | Val Acc: 0.7051 | Val F1: 0.7243
Epoch 6/50 | Loss: 0.6674 | Val Acc: 0.7345 | Val F1: 0.7470
Epoch 7/50 | Loss: 0.6669 | Val Acc: 0.7209 | Val F1: 0.7390
Epoch 8/50 | Loss: 0.6601 | Val Acc: 0.7220 | Val F1: 0.7391
Epoch 9/50 | Loss: 0.6543 | Val Acc: 0.7322 | Val F1: 0.7408
Epoch 10/50 | Loss: 0.6492 | Val Acc: 0.7254 | Val F1: 0.7412
Epoch 11/50 | Loss: 0.6455 | Val Acc: 0.6915 | Val F1: 0.7111
Epoch 12/50 | Loss: 0.6457 | Val Acc: 0.7017 | Val F1: 0.7194
Epoch 13/50 | Loss: 0.6391 | Val Acc: 0.7164 | Val F1: 0.7346
Early stopping di epoch 13
Training Selesai. Best F1 Score Neural ODE: 0.7564
